# 결측치·이상치 분석 준비 (`03_Missing_Outlier`)

이 노트북은 25일 전체 코호트 통합 정본에서 **모델 대상만 추출하고 데이터 계약을 검증**한 뒤,
결측치 원인 진단과 처리, 이상치 점검을 이어서 진행한다.

## 1. 환경 설정과 통합 정본 로드

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'CSV_files').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '../..').resolve()

DATA_PATH = PROJECT_ROOT / 'CSV_files' / '통합 버전' / 'landmark25_all_cohorts.csv'
KEYS = ['code_module', 'code_presentation', 'id_student']

assert DATA_PATH.exists(), f'통합 정본을 찾을 수 없음: {DATA_PATH}'
all_cohorts = pd.read_csv(DATA_PATH)
print('통합 정본:', DATA_PATH)
print('전체 shape:', all_cohorts.shape)

통합 정본: C:\DA_WorkSpace\Personal_OULAD_Churn_Prediction\CSV_files\통합 버전\landmark25_all_cohorts.csv
전체 shape: (32593, 29)


## 2. 모델 대상 추출

25일 시점 재학 상태를 확인할 수 있는 `eligible_at_25 == 1`이면서
`cohort_status_25 == 'model_eligible'`인 행만 선택한다. 원본 통합 CSV는 변경하지 않는다.

In [2]:
model_df = all_cohorts.loc[
    (all_cohorts['eligible_at_25'] == 1)
    & (all_cohorts['cohort_status_25'] == 'model_eligible')
].copy()

model_df.reset_index(drop=True, inplace=True)
model_df.shape

(27661, 29)

## 3. 데이터 계약 검증

행 수, 복합 키 유일성, 타깃 결측·값 범위와 양성 건수를 확인한다.

In [3]:
assert len(all_cohorts) == 32_593, '전체 코호트 행 수 불일치'
assert all_cohorts.duplicated(KEYS).sum() == 0, '전체 코호트 키 중복 발생'
assert len(model_df) == 27_661, '모델 대상 행 수 불일치'
assert model_df.duplicated(KEYS).sum() == 0, '모델 대상 키 중복 발생'
assert model_df['target_churn_after_25'].notna().all(), '모델 대상 타깃 결측 발생'
assert set(model_df['target_churn_after_25'].unique()) <= {0, 1}, '타깃이 0/1 이외의 값을 포함'
assert model_df['target_churn_after_25'].sum() == 5_247, '양성 건수 불일치'
assert model_df['eligible_at_25'].eq(1).all(), '제외 코호트 혼입'
assert model_df['cohort_status_25'].eq('model_eligible').all(), '코호트 상태 불일치'

validation_summary = pd.Series({
    '전체 코호트 행 수': len(all_cohorts),
    '모델 대상 행 수': len(model_df),
    '복합 키 중복': model_df.duplicated(KEYS).sum(),
    '타깃 결측': model_df['target_churn_after_25'].isna().sum(),
    '양성 건수': int(model_df['target_churn_after_25'].sum()),
    '음성 건수': int((model_df['target_churn_after_25'] == 0).sum()),
})
validation_summary

전체 코호트 행 수    32593
모델 대상 행 수     27661
복합 키 중복           0
타깃 결측             0
양성 건수          5247
음성 건수         22414
dtype: int64

## 4. 결측치 진단

`model_df`의 결측 컬럼 5개(`imd_band`, `date_registration`, `avg_score_25`, `avg_submit_delay_25`, `submission_rate_25`)를 대상으로 결측 건수·비율을 확인하고, 각 결측이 구조적 결측(파생 규칙상 필연적으로 발생)인지 일반 결측(원본 데이터의 결측)인지 구분한다.

In [4]:
missing_cols = [
    'imd_band', 'date_registration',
    'avg_score_25', 'avg_submit_delay_25', 'submission_rate_25',
]
missing_summary = pd.DataFrame({
    '결측 건수': model_df[missing_cols].isna().sum(),
    '결측 비율(%)': (model_df[missing_cols].isna().mean() * 100).round(2),
})
missing_summary

,결측 건수,결측 비율(%)
imd_band,1030,3.72
date_registration,7,0.03
avg_score_25,10023,36.24
avg_submit_delay_25,10017,36.21
submission_rate_25,6766,24.46


### 4.1 평가 행동 결측 (`submission_rate_25`, `avg_score_25`, `avg_submit_delay_25`)

세 컬럼 모두 `n_opportunity_25`·`n_submitted_25`에서 파생됐으므로, 결측이 `no_assessment_opportunity_25`(25일까지 마감된 평가 기회 자체가 없음) 플래그 및 `n_submitted_25 == 0`(기회는 있었지만 미제출)과 어떻게 대응하는지 확인한다.

In [5]:
# submission_rate_25: 결측이 '평가 기회 없음'과 정확히 일치하는지 확인
opportunity_crosstab = pd.crosstab(
    model_df['submission_rate_25'].isna().rename('submission_rate_25_결측'),
    model_df['no_assessment_opportunity_25'].rename('no_assessment_opportunity_25'),
)
opportunity_crosstab

no_assessment_opportunity_25,0.0,1.0
submission_rate_25_결측,,
False,20895,0
True,0,6766


In [6]:
# avg_score_25 / avg_submit_delay_25: 결측을 '기회 없음'과 '기회는 있었지만 미제출'로 분해
no_submit = model_df['n_submitted_25'].eq(0)

score_breakdown = pd.crosstab(
    [model_df['avg_score_25'].isna().rename('avg_score_25_결측'), no_submit.rename('미제출(n_submitted_25==0)')],
    model_df['no_assessment_opportunity_25'].rename('no_assessment_opportunity_25'),
)
delay_breakdown = pd.crosstab(
    model_df['avg_submit_delay_25'].isna().rename('avg_submit_delay_25_결측'),
    no_submit.rename('미제출(n_submitted_25==0)'),
)
print(score_breakdown)
print()
print(delay_breakdown)

no_assessment_opportunity_25              0.0   1.0
avg_score_25_결측 미제출(n_submitted_25==0)             
False           False                   17638     0
True            False                       6     0
                True                     3251  6766

미제출(n_submitted_25==0)  False  True 
avg_submit_delay_25_결측              
False                   17644      0
True                        0  10017


In [7]:
# 예외 확인: 제출은 했는데(n_submitted_25 > 0) avg_score_25가 결측인 행
score_edge_cases = model_df.loc[
    (model_df['n_submitted_25'] > 0) & (model_df['avg_score_25'].isna()),
    ['code_module', 'code_presentation', 'id_student', 'n_opportunity_25',
     'n_submitted_25', 'n_missing_25', 'avg_score_25', 'avg_submit_delay_25'],
]
len(score_edge_cases), score_edge_cases

(6,
       code_module code_presentation  id_student  n_opportunity_25  \
 1500          BBB             2013B      534151               1.0   
 4729          BBB             2014B      606501               1.0   
 12272         DDD             2013J      427248               1.0   
 18677         FFF             2013B      174436               1.0   
 19570         FFF             2013B      546164               1.0   
 20159         FFF             2013J      126074               1.0   
 
        n_submitted_25  n_missing_25  avg_score_25  avg_submit_delay_25  
 1500              1.0           0.0           NaN                -12.0  
 4729              1.0           0.0           NaN                 -1.0  
 12272             1.0           0.0           NaN                  0.0  
 18677             1.0           0.0           NaN                 -6.0  
 19570             1.0           0.0           NaN                  3.0  
 20159             1.0           0.0           NaN          

**해석**

- `submission_rate_25` 결측 6,766건은 `no_assessment_opportunity_25 == 1`(=`n_opportunity_25 == 0`)과 100% 일치한다. 완전한 구조적 결측이며, 이미 존재하는 `no_assessment_opportunity_25` 플래그로 원인이 전부 설명된다.
- `avg_score_25`/`avg_submit_delay_25` 결측(각 10,023건/10,017건)은 대부분 `n_submitted_25 == 0`(기회 없음 6,766건 + 기회는 있었으나 미제출 3,251건)로 설명된다. 두 원인 모두 이미 `no_assessment_opportunity_25`, `n_submitted_25` 피처에 드러나 있으므로 구조적 결측이다.
- 예외적으로 `avg_score_25`만 6건이 제출 기록(`n_submitted_25 > 0`)이 있음에도 결측이다. 건수가 극히 적어(0.02%) 전체 결론에 영향을 주지 않으며, 원본 `studentAssessment`의 점수 필드 자체가 비어 있는 개별 사례로 추정된다.

### 4.2 `imd_band` 결측 — 지역별 분포

`imd_band`(거주 지역 빈곤 지수)는 잉글랜드 행정구역 기준 지표라, 잉글랜드 외 지역 학생은 원본 데이터에 애초에 값이 없을 수 있다. 지역(`region`)별 결측 비율로 확인한다.

In [8]:
imd_missing_by_region = (
    model_df.groupby('region')['imd_band']
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
    .rename('결측 비율')
)
imd_missing_by_region

region
North Region            0.437459
Ireland                 0.226244
West Midlands Region    0.018052
South Region            0.016717
Scotland                0.003487
Yorkshire Region        0.002402
South West Region       0.001939
North Western Region    0.001699
East Anglian Region     0.000000
London Region           0.000000
East Midlands Region    0.000000
South East Region       0.000000
Wales                   0.000000
Name: 결측 비율, dtype: float64

**해석**

결측이 `North Region`(43.7%)과 `Ireland`(22.6%)에 집중되고 나머지 11개 지역은 2% 미만(대다수 0%)이다. 두 지역만 결측률이 두드러지는 것은 IMD가 잉글랜드 지표라 북아일랜드·일부 접경 지역 학생에게는 애초에 값이 존재하지 않을 가능성을 시사한다. 즉 무작위 결측이 아니라 지역에 종속된 구조적 결측이며, 결측 자체가 '잉글랜드 외 거주'라는 정보를 담고 있어 임의 대치보다는 별도 범주로 다루는 편이 정보 손실이 적다.

### 4.3 `date_registration` 결측

결측이 27,661건 중 7건(0.03%)으로 극히 적다. 개별 행을 직접 확인한다.

In [9]:
date_registration_missing = model_df.loc[
    model_df['date_registration'].isna(),
    ['code_module', 'code_presentation', 'id_student', 'total_click_25',
     'n_opportunity_25', 'target_churn_after_25'],
]
date_registration_missing

,code_module,code_presentation,id_student,total_click_25,n_opportunity_25,target_churn_after_25
2155,BBB,2013B,630346,0.0,1.0,0.0
10566,CCC,2014J,1777834,0.0,1.0,0.0
11907,DDD,2013B,2707979,0.0,2.0,0.0
11908,DDD,2013B,2710343,0.0,2.0,0.0
14591,DDD,2014B,2710343,0.0,1.0,0.0
16404,EEE,2013J,568751,0.0,0.0,1.0
19992,FFF,2013B,2102658,0.0,1.0,0.0


**해석**

7건은 특정 모듈·학기에 몰려 있지 않고, `total_click_25`나 `n_opportunity_25` 등 다른 피처는 정상적으로 채워져 있다. 건수가 워낙 적어 지역·모듈 패턴을 찾기 어렵고, 원본 `studentRegistration`에 등록일 자체가 기록되지 않은 개별 데이터 누락으로 보인다.

## 5. 처리 방침(제안)

아래는 진단 결과를 바탕으로 한 제안이며, 사용자 확정 전까지 `model_df`에 실제 대치·행 제거는 적용하지 않는다.

| 컬럼 | 결측 성격 | 제안 |
|---|---|---|
| `submission_rate_25` | 완전한 구조적 결측 (기회 없음과 100% 일치) | NaN 유지 + `no_assessment_opportunity_25` 플래그로 모델에 전달 (0으로 채우면 '기회는 있었으나 0% 제출'로 오독될 수 있음) |
| `avg_score_25`, `avg_submit_delay_25` | 구조적 결측 (기회 없음 + 미제출로 전부 설명, 예외 6건 존재) | NaN 유지, `n_submitted_25`(이미 피처)가 원인을 대신 설명하므로 별도 플래그 불필요. 예외 6건은 다른 결측과 동일하게 처리 |
| `imd_band` | 지역 종속 구조적 결측 (잉글랜드 외 지역에 집중) | 'Unknown' 범주 추가(수치형 대치 대신 범주형 결측 그대로 인코딩) |
| `date_registration` | 소수 원본 결측 (7건, 0.03%) | 건수가 미미하므로 해당 7행 제외 또는 중앙값 대치 — 모델 성능에 영향 없을 것으로 예상, 편의상 행 제외 권장 |

이상치 점검(`total_click_25`, `avg_submit_delay_25` 등)과 최종 전처리 파이프라인 구현은 다음 작업에서 진행한다.

## 6. 결측치 처리 적용

5절 방침 중 **데이터셋 단계에서 한 번만 적용하면 되는 두 가지**를 이번 절에서 실제로 반영한다.
나머지 평가 행동 결측(`submission_rate_25`, `avg_score_25`, `avg_submit_delay_25`)은 NaN을
그대로 유지하고, 실제 대치 여부와 방식은 모델별로 다르게 모델링 파이프라인 안에서 처리한다
(근거: `work_process/decisions/preprocessing/0912_01_missing_value_policy.md` 5절).

- `imd_band`: 결측을 `'Unknown'` 범주로 채운다.
- `date_registration`: 결측 7행을 제외한다.

In [10]:
model_df_clean = model_df.copy()

model_df_clean['imd_band'] = model_df_clean['imd_band'].fillna('Unknown')

before_drop = len(model_df_clean)
model_df_clean = model_df_clean.loc[model_df_clean['date_registration'].notna()].copy()
model_df_clean.reset_index(drop=True, inplace=True)
dropped = before_drop - len(model_df_clean)

print(f'imd_band 결측 처리 후 결측 건수: {model_df_clean["imd_band"].isna().sum()}')
print(f'date_registration 결측 제외 행 수: {dropped}')
print(f'처리 후 model_df_clean shape: {model_df_clean.shape}')

imd_band 결측 처리 후 결측 건수: 0
date_registration 결측 제외 행 수: 7
처리 후 model_df_clean shape: (27654, 29)


In [11]:
assert model_df_clean['imd_band'].isna().sum() == 0, 'imd_band 결측 잔존'
assert model_df_clean['date_registration'].isna().sum() == 0, 'date_registration 결측 잔존'
assert model_df_clean.duplicated(KEYS).sum() == 0, '처리 후 키 중복 발생'
assert model_df_clean['target_churn_after_25'].notna().all(), '처리 후 타깃 결측 발생'

remaining_missing = model_df_clean[missing_cols].isna().sum()
print('처리 후 남은 결측(의도적으로 유지, 모델별 파이프라인에서 처리 예정):')
print(remaining_missing)

treatment_summary = pd.Series({
    '처리 전 행 수': len(model_df),
    '처리 후 행 수': len(model_df_clean),
    '제외된 행 수(date_registration 결측)': dropped,
    '처리 전 양성 건수': int(model_df['target_churn_after_25'].sum()),
    '처리 후 양성 건수': int(model_df_clean['target_churn_after_25'].sum()),
})
treatment_summary

처리 후 남은 결측(의도적으로 유지, 모델별 파이프라인에서 처리 예정):
imd_band                   0
date_registration          0
avg_score_25           10016
avg_submit_delay_25    10010
submission_rate_25      6765
dtype: int64


처리 전 행 수                         27661
처리 후 행 수                         27654
제외된 행 수(date_registration 결측)        7
처리 전 양성 건수                        5247
처리 후 양성 건수                        5246
dtype: int64

**해석**

`imd_band` 결측 1,030건은 `'Unknown'` 범주로, `date_registration` 결측 7건은 행 제외로
처리했다. `submission_rate_25`/`avg_score_25`/`avg_submit_delay_25`는 이 노트북에서는 NaN을
그대로 두고, 이후 모델별 `Pipeline`(Logistic/RF는 대치, XGBoost는 NaN 유지)에서 처리한다.
이후 분석·모델링은 `model_df`가 아니라 `model_df_clean`을 기준으로 진행한다.

## 다음 작업

결측 5개 컬럼의 원인 진단, 처리 방침 확정과 `imd_band`/`date_registration` 처리 적용까지
완료했다. 다음 작업은 다음과 같다.

1. 이상치 점검(`total_click_25`, `avg_submit_delay_25` 등)을 이 노트북에서 이어서 진행한다.
2. 확정된 결측·이상치 처리를 반영한 모델별 전처리 파이프라인을 구현한다
   (`work_process/decisions/preprocessing/0912_01_missing_value_policy.md` 5절 참고).